# Sementic Retrieval Models Evaluation

This experiment evaluates the performance of lightweight pretrained text embedding models for semantic retrieval under a CPU-only environment. The objective is to analyse the trade-off between retrieval accuracy and computational efficiency when matching natural language queries with stored object descriptions. In this study, each query is converted into a vector embedding using a pretrained sentence embedding model and compared against embeddings of candidate text descriptions in the database using cosine similarity. The retrieved results are ranked according to similarity scores, and the ranking performance is evaluated using Top-1 retrieval accuracy and Mean Reciprocal Rank (MRR). In addition to retrieval quality, query embedding latency and end-to-end retrieval time are measured to assess the computational efficiency of each model under CPU execution constraints. The goal is to identify the most suitable embedding model for the semantic retrieval component of the proposed object locator system, balancing accurate semantic matching with practical real-time performance.

The evaluation will be based on the following metrics
- Top-1 Retrival accuracy
- Mean Reciprocal Rank (MRR)
- Query Embedding Latency
- End-to-end Retrieval Time

Top-1 retrieval accuracy measures the percentage of queries where the correct result is ranked as the most similar item. Mean Reciprocal Rank (MRR) measures how highly the correct result is ranked by assigning higher scores when the correct item appears closer to the top of the retrieval list. Query embedding latency measures the time required to generate an embedding vector for a given query under CPU execution, while end-to-end retrieval time captures the total time required to process a query, compute its embedding, perform vector similarity search, and return the ranked results.

The models that will be evaluated includes:
- all-MiniLM-L6-v2
- BGE-small-en-v1.5
- E5-small-v2

These models were selected because they represent widely used lightweight sentence embedding architectures designed for semantic retrieval tasks. Each model maintains relatively small parameter sizes while providing strong semantic matching performance, making them suitable for real-time retrieval systems operating under CPU-only constraints.

## MiniLM

The MiniLM model will be initialized to generate text embeddings for both query and database descriptions.

In [ ]:
import pandas as pd
import time
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm
# sentence transformer embedding model
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
# loading the query and database dataset
queries = pd.read_csv("queries.csv")
database = pd.read_csv("database.csv")

# extract text descriptions and corresponding object labels from database
descriptions = database["description"].tolist()
objects = database["object"].tolist()
# loading sentence transformer
print("Loading model:", MODEL_NAME)
model = SentenceTransformer(MODEL_NAME)
warmup = model.encode(["test"])

Loading model: sentence-transformers/all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5255.64it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Database descriptions are converted into vector embeddings using the sentence transformer model, enabling later similarity comparison with query embeddings.

In [ ]:
# generate embeddings for all database descriptions
db_embeddings = model.encode(descriptions)

In [41]:
results = []

Each query is processed by generating its embedding and comparing it against precomputed database embeddings using cosine similarity.

In [ ]:
top1_correct = 0
reciprocal_rank_sum = 0

embedding_times = []
total_times = []

print("\nRunning evaluation...\n")
# iterating through each query
for _, row in tqdm(queries.iterrows(), total=len(queries)):
    # extract query text and correct object label
    query = row["query"]
    correct_object = row["correct_object"]

    # start total time measurement for this query
    start_total = time.time()

    # Measure embedding generation latency
    start_embed = time.time()
    query_embedding = model.encode([query])
    embed_time = time.time() - start_embed
    embedding_times.append(embed_time)

    # compute similarity between query embedding and database embeddings
    sims = cosine_similarity(query_embedding, db_embeddings)[0]

    # ranking database entries by descending similarity
    ranked_indices = sims.argsort()[::-1]

    # Top 1 prediction
    top_idx = ranked_indices[0]
    predicted_object = objects[top_idx]
    # check top 1 correctness
    if predicted_object == correct_object:
        top1_correct += 1

    # Compute rank of the correct object for MRR
    rank = None
    for i, idx in enumerate(ranked_indices):
        if objects[idx] == correct_object:
            rank = i + 1
            break
    # accumulate reciprocal rank if found
    if rank is not None:
        reciprocal_rank_sum += 1 / rank
    # measure total processing time for this query
    total_time = time.time() - start_total
    total_times.append(total_time)


Running evaluation...



100%|██████████| 5/5 [00:00<00:00, 66.48it/s]


Calculations for evaluation metrics

In [43]:
num_queries = len(queries)

top1_accuracy = top1_correct / num_queries
mrr = reciprocal_rank_sum / num_queries
avg_embed_time = sum(embedding_times) / num_queries
avg_total_time = sum(total_times) / num_queries

Putting the results into a dataframe for analysis

In [44]:
results_df = pd.DataFrame(columns=[
    "Model",
    "Top-1 Accuracy",
    "MRR",
    "Query Latency (ms)",
    "End-to-End Time (ms)"
])

In [45]:
results_df.loc[len(results_df)] = [
    MODEL_NAME,
    round(top1_accuracy, 3),
    round(mrr, 3),
    round(avg_embed_time * 1000, 2),
    round(avg_total_time * 1000, 2)
]

In [46]:
results_df

,Model,Top-1 Accuracy,MRR,Query Latency (ms),End-to-End Time (ms)
0,sentence-transformers/all-MiniLM-L6-v2,1.0,1.0,14.24,14.84


## BGE-small-en-v1.5

In [ ]:
import pandas as pd
import time
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from tqdm import tqdm
# sentence transformer 
MODEL_NAME = "BAAI/bge-small-en-v1.5"
# loading the query and database dataset
queries = pd.read_csv("queries.csv")
database = pd.read_csv("database.csv")
# extract text descriptions and corresponding object labels from database
descriptions = database["description"].tolist()
objects = database["object"].tolist()
# loading sentence transformer
print("Loading model:", MODEL_NAME)
model = SentenceTransformer(MODEL_NAME)
warmup = model.encode(["test"])

Loading model: BAAI/bge-small-en-v1.5


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5015.12it/s]
BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Database descriptions are converted into vector embeddings using the sentence transformer model, enabling later similarity comparison with query embeddings.

In [ ]:
# generate embeddings for all database descriptions
db_embeddings = model.encode(descriptions)

Each query is processed by generating its embedding and comparing it against precomputed database embeddings using cosine similarity.

In [ ]:
top1_correct = 0
reciprocal_rank_sum = 0

embedding_times = []
total_times = []

print("\nRunning evaluation...\n")
# iterating through each query
for _, row in tqdm(queries.iterrows(), total=len(queries)):
    # extract query text and correct object label
    query = row["query"]
    correct_object = row["correct_object"]
    # start total time measurement for this query
    start_total = time.time()

    # compute similarity between query embedding and database embeddings
    start_embed = time.time()
    query_embedding = model.encode([query])
    embed_time = time.time() - start_embed
    embedding_times.append(embed_time)

    # compute similarity between query embedding and database embeddings
    sims = cosine_similarity(query_embedding, db_embeddings)[0]
    # ranking database entries by descending similarity
    ranked_indices = sims.argsort()[::-1]

    # Top-1 prediction
    top_idx = ranked_indices[0]
    predicted_object = objects[top_idx]
    # check top 1 correctness
    if predicted_object == correct_object:
        top1_correct += 1

    # Compute rank of the correct object for MRR
    rank = None
    for i, idx in enumerate(ranked_indices):
        if objects[idx] == correct_object:
            rank = i + 1
            break
    # accumulate reciprocal rank if found
    if rank is not None:
        reciprocal_rank_sum += 1 / rank
    # measure total processing time for this query
    total_time = time.time() - start_total
    total_times.append(total_time)


Running evaluation...



100%|██████████| 5/5 [00:00<00:00, 44.79it/s]


Calculations for the evaluation metrics

In [50]:
num_queries = len(queries)

top1_accuracy = top1_correct / num_queries
mrr = reciprocal_rank_sum / num_queries
avg_embed_time = sum(embedding_times) / num_queries
avg_total_time = sum(total_times) / num_queries

Appending the results to the existing database

In [51]:
results_df.loc[len(results_df)] = [
    MODEL_NAME,
    round(top1_accuracy, 3),
    round(mrr, 3),
    round(avg_embed_time * 1000, 2),
    round(avg_total_time * 1000, 2)
]

In [52]:
results_df

,Model,Top-1 Accuracy,MRR,Query Latency (ms),End-to-End Time (ms)
0,sentence-transformers/all-MiniLM-L6-v2,1.0,1.0,14.24,14.84
1,BAAI/bge-small-en-v1.5,1.0,1.0,21.33,21.93


## e5-small-v2

The e5 model willbe initialized to generate text embedding for both query and database descriptions

In [ ]:
# sentence transformer embedding model
MODEL_NAME = "intfloat/e5-small-v2"
# loading the query and database dataset
queries = pd.read_csv("queries.csv")
database = pd.read_csv("database.csv")
# extract text descriptions and corresponding object labels from database
descriptions = database["description"].tolist()
objects = database["object"].tolist()
# loading sentence transformer
print("Loading model:", MODEL_NAME)
model = SentenceTransformer(MODEL_NAME)
warmup = model.encode(["test"])

Loading model: intfloat/e5-small-v2


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5086.98it/s]
BertModel LOAD REPORT from: intfloat/e5-small-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Database descriptions are converted into vector embeddings using the sentence transformer model, enabling later similarity comparison with query embeddings.

In [ ]:
# generate embeddings for all database descriptions
db_embeddings = model.encode(descriptions)

Each query is processed by generating its embedding and comparing it against precomputed database embeddings using cosine similarity.

In [ ]:
top1_correct = 0
reciprocal_rank_sum = 0

embedding_times = []
total_times = []

print("\nRunning evaluation...\n")
# iterating through each query
for _, row in tqdm(queries.iterrows(), total=len(queries)):
    # extract query text and correct object label
    query = row["query"]
    correct_object = row["correct_object"]
    # start total time measurement for this query
    start_total = time.time()

    # Measure embedding generation latency
    start_embed = time.time()
    query_embedding = model.encode([query])
    embed_time = time.time() - start_embed
    embedding_times.append(embed_time)

    # compute similarity between query embedding and database embeddings
    sims = cosine_similarity(query_embedding, db_embeddings)[0]
    # ranking database entries by descending similarity
    ranked_indices = sims.argsort()[::-1]

    # Top 1 prediction
    top_idx = ranked_indices[0]
    predicted_object = objects[top_idx]
    # check top 1 correctness
    if predicted_object == correct_object:
        top1_correct += 1

    # Compute rank of the correct object for MRR
    rank = None
    for i, idx in enumerate(ranked_indices):
        if objects[idx] == correct_object:
            rank = i + 1
            break
    # accumulate reciprocal rank if found
    if rank is not None:
        reciprocal_rank_sum += 1 / rank
    # measure total processing time for this query
    total_time = time.time() - start_total
    total_times.append(total_time)


Running evaluation...



100%|██████████| 5/5 [00:00<00:00, 38.37it/s]


Calculations for evaluation metrics

In [56]:
num_queries = len(queries)

top1_accuracy = top1_correct / num_queries
mrr = reciprocal_rank_sum / num_queries
avg_embed_time = sum(embedding_times) / num_queries
avg_total_time = sum(total_times) / num_queries

Putting the results into a dataframe for analysis

In [57]:
results_df.loc[len(results_df)] = [
    MODEL_NAME,
    round(top1_accuracy, 3),
    round(mrr, 3),
    round(avg_embed_time * 1000, 2),
    round(avg_total_time * 1000, 2)
]

In [58]:
results_df

,Model,Top-1 Accuracy,MRR,Query Latency (ms),End-to-End Time (ms)
0,sentence-transformers/all-MiniLM-L6-v2,1.0,1.0,14.24,14.84
1,BAAI/bge-small-en-v1.5,1.0,1.0,21.33,21.93
2,intfloat/e5-small-v2,1.0,1.0,24.78,25.66


All three text embedding models achieved perfect Top-1 retrieval accuracy and MRR on the evaluation dataset. This indicates that the retrieval task, as defined by the current set of object descriptions and queries, was relatively simple and did not sufficiently challenge the semantic discrimination capability of the models. As a result, computational efficiency became the main differentiating factor, with all-MiniLM-L6-v2 showing the lowest query latency and end-to-end retrieval time under CPU-only execution.